# 01 — Exploratory Data Analysis & Data Quality

**Project FORESIGHT · NorthBay Living**

Purpose: understand what the client actually gave us before modelling anything.
Three questions drive this notebook:

1. Is the data trustworthy? (what is broken, and how did we handle it)
2. What shape does demand have? (trend, seasonality, promotions, concentration)
3. What does that imply for how we forecast?

Run `python run_all.py` from the repo root first, so the processed files exist.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)


## 1. Raw extracts — first look

Profile the extracts before any cleaning, so we can see the defects.

In [ ]:
raw_sales = pd.read_csv(C.RAW_DIR / "sales_daily.csv")
raw_skus = pd.read_csv(C.RAW_DIR / "sku_master.csv")
raw_inv = pd.read_csv(C.RAW_DIR / "inventory_snapshots.csv")

print("sales_daily        ", raw_sales.shape)
print("sku_master         ", raw_skus.shape)
print("inventory_snapshots", raw_inv.shape)
raw_sales.head()

In [ ]:
# Where are the defects?
print("Missing values in sales_daily:")
print(raw_sales.isna().sum()[lambda s: s > 0])
print("\nExact duplicate rows:", raw_sales.duplicated().sum())
print("Rows with negative units:", (raw_sales["units_sold"] < 0).sum())
print("\nCategory labels before cleaning:")
print(raw_skus["category"].value_counts())

The category column shows the same categories appearing multiple times with
different casing and stray whitespace. Left alone this would split each category
into several fake categories and corrupt every category-level feature.

## 2. Run the cleaning pipeline and read its audit log

In [ ]:
import json

from src import pipeline as P

out = P.run()
weekly, skus = out["weekly"], out["skus"]

print(json.dumps(out["log"]["sales_daily"], indent=2))

In [ ]:
print("Category labels after cleaning:")
print(skus["category"].value_counts())
print("\nWeekly grain:", weekly.shape)
weekly.head()

### Why zero-filling matters

A week with no sales row means zero demand, not missing demand. If the gap is
left in place, `lag_1` silently reaches back further than one week for exactly
the SKUs that are behaving unusually — the ones we most need to get right.

## 3. Demand shape

In [ ]:
total = weekly.groupby("week_start", as_index=False)["units_sold"].sum()

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(total["week_start"], total["units_sold"], lw=1.8, color="#6b5bd6")
ax.set_title("Total weekly demand, all SKUs")
ax.set_ylabel("Units / week")
plt.show()

In [ ]:
# Seasonality: average demand per SKU by week of year
w = weekly.copy()
w["woy"] = w["week_start"].dt.isocalendar().week.astype(int)
by_woy = w.groupby("woy")["units_sold"].mean()

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(by_woy.index, by_woy.values, color="#6b5bd6", alpha=0.85)
ax.set_title("Mean units per SKU by week of year")
ax.set_xlabel("Week of year")
plt.show()

print(f"Peak week {by_woy.idxmax()} runs {by_woy.max() / by_woy.min():.1f}x the trough (week {by_woy.idxmin()})")

In [ ]:
# Revenue concentration — where should planning effort go?
by_sku = weekly.groupby("sku_id")["revenue"].sum().sort_values(ascending=False)
cum = by_sku.cumsum() / by_sku.sum()
top20 = int(len(by_sku) * 0.2)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(range(1, len(cum) + 1), cum.values * 100, lw=2, color="#6b5bd6")
ax.axvline(top20, ls="--", color="#d69b45")
ax.set_title("Cumulative revenue share by SKU rank")
ax.set_ylabel("Cumulative revenue %")
plt.show()

print(f"Top 20% of SKUs = {cum.iloc[top20 - 1] * 100:.0f}% of revenue")

In [ ]:
# Promotion effect
promo = weekly[weekly["promo_flag"] == 1]["units_sold"].mean()
base = weekly[weekly["promo_flag"] == 0]["units_sold"].mean()
print(f"Normal week: {base:.2f} units/SKU")
print(f"Promo week : {promo:.2f} units/SKU  ({(promo / base - 1) * 100:.0f}% lift)")

In [ ]:
# Intermittency — does WAPE or MAPE make sense here?
zero_share = (weekly["units_sold"] == 0).mean()
cv = weekly.groupby("sku_id")["units_sold"].agg(["mean", "std"])
cv["cv"] = cv["std"] / cv["mean"].replace(0, np.nan)

print(f"Share of SKU-weeks with zero demand: {zero_share * 100:.1f}%")
print(f"Median SKU coefficient of variation: {cv['cv'].median():.2f}")
print(f"SKUs with CV > 1 (highly volatile) : {(cv['cv'] > 1).mean() * 100:.0f}%")

**This settles the metric choice.** With zero-demand weeks present, MAPE is
undefined or explodes on exactly those rows. WAPE — total absolute error divided
by total actual demand — stays well-defined and weights SKUs by how much they
actually sell, which matches how the business experiences error.

## 4. What this means for modelling

| Finding | Modelling consequence |
|---|---|
| Strong annual seasonality (peak ≈ 2.7× trough) | Must include a same-week-last-year anchor and cyclical calendar features |
| Promotions lift demand ~80% | Promo flag must be a feature; NorthBay knows its own calendar ahead of time |
| Revenue highly concentrated | Weight attention and error review by revenue, not by SKU count |
| Lumpy, low-volume SKUs | WAPE over MAPE; prediction intervals matter more than point estimates |
| Some SKUs have thin history | Favour one pooled global model over 200 per-SKU models |
